In [26]:
import gzip
import struct
import pandas as pd

------Setting up the Conditions-------

pandas can read specific table format data like csv or excel.
But for files like binary compressed, we will need gzip to open or decompress such file.
Similarly, "Struct" will decode or translate the binary bites to Python Values.
And finally, we can use pandas on it to present and analyse from DataFrame, Table, form.

As per the qsn, the values are stored in big-endian (network) byte order i.e. > I I I I B B B B

4I = 4*4 = 16 bytes

4B = 4*1 = 4 bytes

--------

         = 20 bytes
         

In [27]:
requests = []
# with gzip, open .gz file so as to read its binary data (rb)
with gzip.open("TR1.gz","rb") as f:
    while True:
        data = f.read(20) # read data only upto 20 bytes
        if len (data) < 20:
            break
        # unpacking tuple by assigning variable to each items
        timestamp, client, obj, size, method, status, file_type, server = \
             struct.unpack (">IIIIBBBB",data)
        requests.append([
            timestamp, client, obj, size, method, status, file_type, server]
            )

In [28]:
#creating the dataframe of those converted and appended data
df = pd.DataFrame(requests,columns =[
    "Timestamp",
    "client_id",
    "Object_id",
    "Size",
    "Method",
    "Status",
    "Type",
    "Server"
    ])
#create new column "datetime" in df
df["datetime"] = pd.to_datetime(
    df["Timestamp"],
    unit = "s",
    utc = True   # create universal time regardless of local
) # conversion to readable time from unixTimestamp

#sort all request rows by Timestamp and reset the index
df = df.sort_values("Timestamp").reset_index(drop =True)
print (df.head())
print (df.shape)









   Timestamp  client_id  Object_id   Size  Method  Status  Type  Server  \
0  895615201      14275       6219    546       0      66     1      99   
1  895615201       3817        233    631       0      66     1      99   
2  895615201     385403        403    993       0      66     1      99   
3  895615201      28686        981   3717       0      66     1      99   
4  895615201     385366        635  16290       0      66     1      99   

                   datetime  
0 1998-05-19 22:00:01+00:00  
1 1998-05-19 22:00:01+00:00  
2 1998-05-19 22:00:01+00:00  
3 1998-05-19 22:00:01+00:00  
4 1998-05-19 22:00:01+00:00  
(3041727, 9)


In [29]:
#creating the variables

df["Interarrival_time"] = df["Timestamp"].diff()
df["minute"] = df["datetime"].dt.floor("min")   #places each request by its min. remove seconds
#group requests by minute and count how many requests occurred in each minute
request_per_min = df.groupby("minute").size().reset_index(name = "request_count")
request_per_min

,minute,request_count
0,1998-05-19 22:00:00+00:00,1569
1,1998-05-19 22:01:00+00:00,1981
2,1998-05-19 22:02:00+00:00,1957
3,1998-05-19 22:03:00+00:00,1642
4,1998-05-19 22:04:00+00:00,1580
...,...,...
1435,1998-05-20 21:55:00+00:00,2447
1436,1998-05-20 21:56:00+00:00,2255
1437,1998-05-20 21:57:00+00:00,2214
1438,1998-05-20 21:58:00+00:00,2244


PART 1:

In [30]:
print (df["Type"].value_counts())
df["Size"].describe()

Type
1     2599255
0      357362
10      47066
4       26098
12       5951
8        4468
3         839
7         484
6         100
2          98
9           4
5           2
Name: count, dtype: int64


,Size
count,3.041727e+06
mean,6.908754e+08
std,1.577959e+09
min,0.000000e+00
25%,6.460000e+02
50%,1.431000e+03
75%,1.031200e+04
max,4.294967e+09


PART 1.1

In [31]:
import numpy as np
A = df["Type"]==1 #event where the requests has type 1
B = df["Size"]> 5000 #events where the requests has size > 5000
P_A = A.mean()
P_B = B.mean()
P_A_n_B = (A&B).mean()
P_A_U_B = P_A + P_B - P_A_n_B
Union = (A|B).mean()

print ("The probabilty of event be Type 1 = ", P_A)

print ("The probability of event having size > 5000 = ", P_B)

print ("The probability of event co-exist together = ", P_A_n_B)

print ("The probability of event either existing = ", P_A_U_B)

np.isclose(P_A_U_B,Union) # checking numeric formula with direct formula



The probabilty of event be Type 1 =  0.8545326388594374
The probability of event having size > 5000 =  0.3256666360919307
The probability of event co-exist together =  0.23399470103661504
The probability of event either existing =  0.9462045739147531


np.True_

PART 1.2

Given the situation, we already know the event has size greater than 5000; What would be the probability of event having Type 1.
Here, we narrowed down our search from entire event to only the event satisfying the condition.
i.e. P(A|B) = P(AnB)/p(B)

AND VICE-VERSA

In [32]:
P_A_Given_B= (P_A_n_B)/ P_B     #p(type1|size>5000)
p_B_Given_A= P_A_n_B / P_A      #p(size>5000|type1)

print ("The probability of request having Type 1, Given the selected sizes are greater than 5000 = ", P_A_Given_B)
print ("The probabilty of request having size > 5000, Given the selected requests have TYPE 1 = ", p_B_Given_A)

The probability of request having Type 1, Given the selected sizes are greater than 5000 =  0.7185098966372531
The probabilty of request having size > 5000, Given the selected requests have TYPE 1 =  0.273827692935091


1.3

We can infere that the event A and B are not independent.
The probability of request be Type 1 reduced from 85.45% to 71.85% when it has size greater than 5000.
similarly, the overall probability of size be greater than 5000 was approx. 32.56% among all requests, but the very probability droppped to 27.38% when the event has only Type 1.
Since knowing one event changes the probability of the other event, the two events are not independent.

In [33]:
#seperating the requests in different types
T1 = (df["Type"] == 1)
T0 = (df["Type"]==0)
T10 = (df["Type"] ==10)
T4 = (df["Type"]==4)
others = ~ df["Type"].isin([1,0,10,4]) # remaining requests having rest of others type
#setting up the conditiion to find prob of size > 5000 , given different types
def check_size(given):
    return (B & given).mean()/given.mean()

all_type = [T1,T0,T10,T4,others]
P_size_great_than_5000 = 0
for x in all_type:
 P_size_great_than_5000 += check_size(x)* x.mean() # multiplying likely of request having size > 5000 in each type with the corresponding weight of probability of that type
print ("The total probabilty of request having size > 5000 = " , P_size_great_than_5000)

np.isclose(P_B,P_size_great_than_5000) # comparing both the results


The total probabilty of request having size > 5000 =  0.32566663609193064


np.True_

1.4
here, we grouped different requests by its type, then checked the likely of size having greater than 5k given each type.

then we multiplied it by the weight of that type be occuring in total request.

adding it all together, we got the total probability of the request having size greater than 5000.

Conclusion:

The outcome matches to earlier probability of size  greather than 5k from our observed calculation.

ie.

P_B is appprox to P_size_great_than_5000


In [34]:

# we already have this parameter
A = df["Type"]==1 #event where the requests has type 1
B = df["Size"]> 5000 #events where the requests has size > 5000
P_A_Given_B= (P_A_n_B)/ P_B
p_B_Given_A= P_A_n_B / P_A

#Bayes' calculation
Bayes_A_Given_B = (p_B_Given_A * P_A) / P_B

print(Bayes_A_Given_B)
print(P_A_Given_B)

np.isclose(Bayes_A_Given_B, P_A_Given_B)



0.7185098966372531
0.7185098966372531


np.True_

Using Bayes' theorem, the probability that a request is Type 1 given that its size is greater than 5000 was approximately 71.85%. This matched the conditional probability measured directly from the dataset, confirming the Bayes calculation.

PART 2

QSN 6:

In [35]:
#setting up random variable
request_per_min = df.groupby("minute").size().reset_index(name = "request_count")
X = request_per_min["request_count"]
total_minute = len (request_per_min)
pmf_table = X.value_counts().reset_index(name = "Frequency")
#loop through each frequncy to find its probabilty
P_x = []
for prob in pmf_table["Frequency"]:
    pmf_each= prob/total_minute
    P_x.append(pmf_each)
pmf_table["P(X=x)"] = P_x
pmf_table = pmf_table.sort_values("request_count").reset_index(drop = True) # sorting the value by request_count per min
pmf_table

,request_count,Frequency,P(X=x)
0,784,1,0.000694
1,856,1,0.000694
2,881,1,0.000694
3,886,1,0.000694
4,892,1,0.000694
...,...,...,...
1043,3670,1,0.000694
1044,3672,1,0.000694
1045,4074,1,0.000694
1046,6786,1,0.000694


We separated the requests into one-minute intervals and used the number of requests in each minute as our random variable X. Then we calculated how frequently each value of X occurred. Finally, we divided each frequency by the total number of minutes to calculate P(X=x).


QSN 7.

In [36]:
cdf_prob = []
count_sum = 0
for cdf in pmf_table["P(X=x)"]:
    count_sum +=cdf
    cdf_prob.append(count_sum)
pmf_table["P(X<=x)"] = cdf_prob
pmf_table


,request_count,Frequency,P(X=x),P(X<=x)
0,784,1,0.000694,0.000694
1,856,1,0.000694,0.001389
2,881,1,0.000694,0.002083
3,886,1,0.000694,0.002778
4,892,1,0.000694,0.003472
...,...,...,...,...
1043,3670,1,0.000694,0.997222
1044,3672,1,0.000694,0.997917
1045,4074,1,0.000694,0.998611
1046,6786,1,0.000694,0.999306


CDF(x) is bascially  the sum of pmf upto x

QSN 8.

In [37]:
sample_mean = X.mean()
variance = X.var()
print ("The sample mean = ", sample_mean)
print ("The variance from mean = ", variance)

#finding mean by its expected value
expected_mean = 0
for i in range (len (pmf_table)):
    freq = pmf_table["request_count"][i]
    weight = pmf_table["P(X=x)"][i]
    expected_mean += freq*weight
print ("The expected value of mean = ", expected_mean)
np.isclose(sample_mean,expected_mean)


The sample mean =  2112.3104166666667
The variance from mean =  449858.25034311943
The expected value of mean =  2112.3104166666676


np.True_

The sample mean is 2112.31 which means on average, The server received approx 2112 requests every minute.
Similary, the indivdual requests are seperated by the variance of approx 449858 from the mean.

QSN 9:


In [ ]:
!pip install scipy

In [43]:
from scipy.stats import poisson
lambda_poisson = sample_mean
pois_table = []

 # caclulating poisson for each min
for poiss in pmf_table["request_count"]:
    poiss_pmf = poisson.pmf(poiss,lambda_poisson)
    pois_table.append(poiss_pmf)
pmf_table["Poisson(P(X=k))"] = pois_table
pmf_table.head(8)







,request_count,Frequency,P(X=x),P(X<=x),Poisson(P(X=k))
0,784,1,0.000694,0.000694,5.522011e-242
1,856,1,0.000694,0.001389,2.094271e-212
2,881,1,0.000694,0.002083,9.222652e-203
3,886,1,0.000694,0.002778,7.184325e-201
4,892,1,0.000694,0.003472,1.288432e-198
5,895,1,0.000694,0.004167,1.699506e-197
6,906,1,0.000694,0.004861,1.998548e-193
7,909,1,0.000694,0.005556,2.516117e-192


The Poisson model and our observed data have the same mean (2112.31) because we used the sample mean as the value of λ. However, a Poisson distribution assumes that the variance should be close to the mean (λ). In our observed data, the variance is 449858.25, which is much higher than the Poisson variance of 2112.31. This shows that our request data has much more variation than what a normal Poisson model expects. Also, when we compare the Poisson probabilities with our actual observed probabilities, they are significantly different. Therefore, the basic Poisson model may not be a good fit for representing this request traffic data.

QSN 10.


In [44]:
from math import comb
B = df["Size"]>5000
n = 10 # number of trials
P_B = B.mean()
x = np.arange(0,n+1)
pmf = np.array(comb(n,K)*(P_B)**K * (1-(P_B))**(n-K) for K in x)
bino_table = pd.DataFrame(
    {"X=k": x,
    "P(X=k)" : pmf})
print ("The probability for each possible number of successes in 10 trials : ")
print (bino_table)
# for no event to be true, we have K = 0
K = 0
K_0 = comb(n,K) * (P_B)**K * (1-(P_B))**(n-K)
print (f"The probability of no success at all in {n} trial = ", K_0)

# probability of atleast one event means all the possible success other than the likely of occuring 0, i.e.
NO_0   = 1-K_0
print ("The likely of atleast one success occured = ", NO_0)

The probability for each possible number of successes in 10 trials : 
    X=k    P(X=k)
0     0  0.019442
1     1  0.093896
2     2  0.204059
3     3  0.262799
4     4  0.222106
5     5  0.128718
6     6  0.051803
7     7  0.014296
8     8  0.002589
9     9  0.000278
10   10  0.000013
The probability of no success at all in 10 trial =  0.019442247948616886
The likely of atleast one success occured =  0.9805577520513831


The Binomial distribution lets us find the probability of a certain number of successes occurring in a given number of trials.

QSN 11

In [45]:
# wip
def display_distribution_table(y_values, pmf_values):
    """Return a table with Y, PMF, and CDF."""
    df = pd.DataFrame({
        "y": y_values,
        "P(Y=y)": pmf_values
    })
    df["F(y)=P(Y<=y)"] = df["P(Y=y)"].cumsum()
    return df

Y =  np.arange(0, n+1) # number of requests until first success

pmf = np.array([(1 - P_B)**(k - 1) * P_B for k in Y])

geometric_table = display_distribution_table(Y, pmf)
geometric_table

,y,P(Y=y),F(y)=P(Y<=y)
0,0,0.482946,0.482946
1,1,0.325667,0.808613
2,2,0.219608,1.028221
3,3,0.148089,1.176309
4,4,0.099861,1.276171
5,5,0.067340,1.343511
6,6,0.045409,1.388920
7,7,0.030621,1.419541
8,8,0.020649,1.440190
9,9,0.013924,1.454114


(talk about theoretical/empirical means)

QSN 12

In [46]:
# wip
r = 3
Y = np.arange(0, n+1)
pmf = np.array([comb((k-1), (r-1)) * (P_B**r) * ((1 - P_B)**(k - r)) for k in Y])
binomial_table = display_distribution_table(Y, pmf)
binomial_table
print("")

ValueError: n must be a non-negative integer